# 风险数据模型

风险数据模型采用了三层结构：
* 最上层是**风险库**：`QuantStudio.Risk.RiskDB.RiskDB`
* 风险库包含多张**风险表**：`QuantStudio.Risk.RiskTable.RiskTable`
* 每张风险表中又包含多个时点，每个时点存储的是风险数据

风险模型分为两类：
* **无结构风险模型**：风险矩阵就是协方差矩阵，是以证券代码索引的二维矩阵，通过历史收益率直接计算得出。使用 `RiskDB` + `RiskTable` 存取。
* **多因子风险模型**：基于结构化多因子分解的风险矩阵，可拆解为因子风险矩阵、因子暴露和特异性风险。使用 `FactorRDB` + `FactorRT` 存取。

## 数据组织

每张风险表的数据逻辑上是一个三维数组，第一维是时点，第二维是证券代码，第三维也是证券代码。具体到程序里的数据类型，以 `Panel`（`QuantStudio.Core.QSObject.Panel`）数据类型组织。QuantStudio 规定了这三个维度的先后顺序，对应于 Panel 数据类型，items 是时点，major_axis 是证券代码，minor_axis 是证券代码。在 QuantStudio 的所有 API 中，凡是涉及到风险数据的地方，都将遵守此组织原则。

另外，对于时间点，采用 Python 的 datetime 表示；证券代码的数据类型为字符串，例如 "000001.SZ"。

```mermaid
graph TB
    subgraph RiskFramework["风险数据框架"]
        direction TB

        RiskDB["风险库"]

        RiskTable1["风险表1"]
        RiskTable2["风险表2"]

        subgraph RT1["风险表1内容"]
            F11["时点1"]
            F12["时点2"]
            F1M["时点M"]
        end

        subgraph RT2["风险表2内容"]
            F21["时点1"]
            F22["时点2"]
            F2N["时点N"]
        end
        
    end
    
    RiskDB --> RiskTable1
    RiskDB --> RiskTable2
    
    RiskTable1 --> F11
    RiskTable1 --> F12
    RiskTable1 --> F1M
    
    RiskTable2 --> F21
    RiskTable2 --> F22
    RiskTable2 --> F2N
    
    DF["风险矩阵 DataFrame<br/>- index: 证券代码<br/>- columns: 证券代码"]
    
    F11 --> DF
    F12 --> DF
    F1M --> DF
    F21 --> DF
    F22 --> DF
    F2N --> DF

    style RiskFramework fill:#e1f5fe
    style RiskDB fill:#f87f89
    style RiskTable1 fill:#efbaf7
    style RiskTable2 fill:#efbaf7
    style DF fill:#fff3e0
```

## 多因子风险数据模型

多因子风险模型将证券收益率的协方差矩阵 $\mathbf{V}$ 分解为：

$$\mathbf{V} = \mathbf{X} \cdot \mathbf{F} \cdot \mathbf{X}^T + \mathbf{\Delta}$$

其中：
- $\mathbf{X}$：因子暴露矩阵（$N \times K$），即因子截面数据
- $\mathbf{F}$：因子收益率协方差矩阵（$K \times K$），即因子风险矩阵
- $\mathbf{\Delta}$：特异性风险对角矩阵（$N \times N$）

因此结构化的风险数据库不仅要存储最终的协方差矩阵，还需要存储构造该矩阵的各个组件。

```mermaid
graph TB
    subgraph MultiFactorRiskFramework["多因子风险数据框架"]
        direction TB

        RiskDB["风险库"]

        RiskTable1["风险表1"]
        RiskTable2["风险表2"]

        subgraph RT1["风险表1内容"]
            F11["时点1"]
            F12["时点2"]
            F1M["时点M"]
        end

        subgraph RT2["风险表2内容"]
            F21["时点1"]
            F22["时点2"]
            F2N["时点N"]
        end
        
    end
    
    RiskDB --> RiskTable1
    RiskDB --> RiskTable2
    
    RiskTable1 --> F11
    RiskTable1 --> F12
    RiskTable1 --> F1M
    
    RiskTable2 --> F21
    RiskTable2 --> F22
    RiskTable2 --> F2N
    
    RiskDataSet["风险数据集"]
    
    F11 --> RiskDataSet
    F12 --> RiskDataSet
    F1M --> RiskDataSet
    F21 --> RiskDataSet
    F22 --> RiskDataSet
    F2N --> RiskDataSet

    CovMatrix["证券协方差阵 V<br/>Panel(items=dts, major=ids, minor=ids)"]
    FactorCovMatrix["因子协方差阵 F<br/>Panel(items=dts, major=因子, minor=因子)"]
    FactorData["因子暴露 X<br/>Panel(items=因子, major=dts, minor=ids)"]
    SpecificRisk["特异性风险 Δ<br/>DataFrame(index=dts, columns=ids)"]
    FactorReturn["因子收益率<br/>DataFrame(index=dts, columns=因子)"]
    SpecificReturn["特异性收益率<br/>DataFrame(index=dts, columns=ids)"]

    RiskDataSet --> CovMatrix
    RiskDataSet --> FactorCovMatrix
    RiskDataSet --> FactorData
    RiskDataSet --> SpecificRisk
    RiskDataSet --> FactorReturn
    RiskDataSet --> SpecificReturn

    style MultiFactorRiskFramework fill:#e1f5fe
    style RiskDB fill:#f87f89
    style RiskTable1 fill:#efbaf7
    style RiskTable2 fill:#efbaf7
    style RiskDataSet fill:#fff3e0
```

# 核心 API 概览

## 风险库（RiskDB）

```python
class RiskDB(__QS_Object__):
    def connect(self) -> Self            # 连接到数据源
    def disconnect(self) -> int          # 断开连接
    def TableNames(self) -> List[str]    # 获取风险表名称列表
    def getTable(table_name, args)       # 获取风险表对象

    # 写入与管理
    def writeData(table_name, idt, icov) # 写入风险数据
    def setTableMetaData(table_name, key, value, meta_data)  # 设置元信息
    def renameTable(old_table_name, new_table_name)          # 重命名表
    def deleteTable(table_name)          # 删除表
    def deleteDateTime(table_name, dts)  # 删除指定时点的数据
```

`FactorRDB` 继承自 `RiskDB`，扩展了多因子风险数据的支持。其 `writeData` 接受 `factor_data`、`factor_cov`、`specific_risk`、`factor_ret`、`specific_ret` 五个可选参数。

## 风险表（RiskTable）

```python
class RiskTable(Node):
    def RiskDB(self) -> RiskDB            # 所属风险库
    def getMetaData(key)                  # 获取元信息
    def getDateTime(start_dt, end_dt)     # 获取时点序列
    def getID(idt)                        # 获取 ID 序列
    def readCov(dts, ids)                 # 读取风险矩阵，返回 Panel
```

RiskTable 还实现了 `__getitem__`，支持 `rt[dt, id]` 的索引方式取协方差值。

`FactorRT` 继承自 `RiskTable`，扩展了多因子风险数据的读取：

```python
class FactorRT(RiskTable):
    def FactorNames(self) -> List[str]           # 因子名称列表
    def getFactorReturnDateTime(start_dt, end_dt)# 因子收益时点序列
    def getSpecificReturnDateTime(start_dt, end_dt)# 特异性收益时点序列

    def readFactorCov(dts)                # 读取因子协方差阵，返回 Panel
    def readFactorData(dts, ids)          # 读取因子暴露，返回 Panel
    def readSpecificRisk(dts, ids)        # 读取特异性风险，返回 DataFrame
    def readFactorReturn(dts)             # 读取因子收益率，返回 DataFrame
    def readSpecificReturn(dts, ids)      # 读取特异性收益率，返回 DataFrame
    def readData(data_item, dts)          # 读取通用数据项
```

## 可用风险库

| 风险库 | 模块 | 存储方式 | 可写 |
|--------|------|----------|------|
| `HDF5RDB` | `QuantStudio.Risk.HDF5RDB` | 本地 HDF5 文件 | 是 |
| `HDF5FRDB` | `QuantStudio.Risk.HDF5RDB` | 本地 HDF5 文件（多因子） | 是 |

## 风险模型

`BarraModel`（`QuantStudio.Risk.RiskModel.BarraModel`）是目前内置的风险模型实现，参考 Barra [CNE5] 多因子风险模型方法论，支持：

1. **因子收益率和特异性收益率估计**：基于截面加权回归（EUE3 方法论）
2. **因子协方差矩阵估计**：EWMA + Newey-West 自相关修正 + Eigenfactor Risk Adjustment + Volatility Regime Adjustment（CHE2 方法论）
3. **特异性风险估计**：EWMA + 结构化模型 + Bayesian Shrinkage + Volatility Regime Adjustment（EUE3 方法论）

```python
class BarraModel:
    __init__(name, factor_table, risk_db, table_name, config_file)
    def setRegressDateTime(dts)     # 设置截面回归时点
    def setRiskESTDateTime(dts)     # 设置风险估计时点
    def run()                       # 执行风险数据生成
```

# 相关文档索引

| 文档 | 内容 |
|------|------|
| [数据读写](数据读写.ipynb) | 风险库的创建、连接、数据读写与元信息管理 |
| [风险模型](风险模型.ipynb) | Barra 多因子风险模型的理论基础与方法论详解 |
| [计算图框架](../Core/计算图框架.ipynb) | 计算图节点、Context、生命周期 |